# 3.4 Combined model confidence and pyDock score

This notebook consolidates AlphaFold2/ColabFold and local AlphaFold3 predictions with their pyDock energies. It ranks models without requiring an experimental reference:

**combined_score = model_confidence_z - pydock_1vdw_z**

Higher model confidence is favourable, whereas lower pyDock-1VDW energy is favourable. The example configuration uses case 4POU.


## 1. Configuration and repository paths

The default cohort includes unique unrelaxed AF2 recycle models and local AF3 models as predicted. Relaxed AF2 models can be enabled later without changing the readers.

Set REPO_ROOT_OVERRIDE to an absolute path, or define the AF_PYDOCK_REPO_ROOT environment variable, when the current working directory is outside this repository.


In [ ]:
import configparser
import hashlib
import json
import os
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

CASE_ID = "4POU"

INCLUDE_AF2_UNRELAXED = True
INCLUDE_AF2_RELAXED = False
INCLUDE_AF3_AS_PREDICTED = True
DROP_DUPLICATE_AF2_FINALS = True
GENERATE_PLOTS = True

# Example manual override:
# REPO_ROOT_OVERRIDE = Path("/absolute/path/to/AF_pyDock")
REPO_ROOT_OVERRIDE = None


In [ ]:
REPO_MARKERS = (
    "Case_Studies_Included",
    "3.4_Combined_Model_Confidence_and_pyDock_Score",
)

def find_repo_root(override=None):
    '''Find the repository from an override, an environment variable, or cwd.'''
    starts = []
    if override is not None:
        starts.append(Path(override).expanduser())
    env_root = os.environ.get("AF_PYDOCK_REPO_ROOT")
    if env_root:
        starts.append(Path(env_root).expanduser())
    starts.append(Path.cwd())

    checked = []
    for start in starts:
        start = start.resolve()
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.append(candidate)
            if all((candidate / marker).exists() for marker in REPO_MARKERS):
                return candidate

    raise FileNotFoundError(
        "Repository root not found. Set REPO_ROOT_OVERRIDE or "
        "AF_PYDOCK_REPO_ROOT to the AF_pyDock repository."
    )

REPO_ROOT = find_repo_root(REPO_ROOT_OVERRIDE)
CASE_DIR = REPO_ROOT / "Case_Studies_Included" / CASE_ID
AF2_DIR = CASE_DIR / "AF2"
AF3_DIR = CASE_DIR / "AF3"
AF3_RESULTS_DIR = AF3_DIR / "output" / CASE_ID
ANALYSIS_DIR = CASE_DIR / "Analysis"

if not CASE_DIR.is_dir():
    raise FileNotFoundError(f"Case directory not found: {CASE_DIR}")
if INCLUDE_AF2_UNRELAXED or INCLUDE_AF2_RELAXED:
    if not AF2_DIR.is_dir():
        print(f"Warning: AF2 directory not found: {AF2_DIR}")
if INCLUDE_AF3_AS_PREDICTED and not AF3_RESULTS_DIR.is_dir():
    print(f"Warning: local AF3 results directory not found: {AF3_RESULTS_DIR}")

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root : {REPO_ROOT}")
print(f"Case directory  : {CASE_DIR}")
print(f"Analysis output : {ANALYSIS_DIR}")


## 2. Input inventory

Only primary model energies are considered. Auxiliary pyDock receptor and ligand PDB files are not model candidates. AF3 confidence discovery is restricted to sample directories, so the aggregate root summary cannot duplicate sample 0.


In [ ]:
AF2_ENERGY_FILES = sorted(AF2_DIR.rglob("*.ene")) if AF2_DIR.is_dir() else []
AF3_ENERGY_FILES = (
    sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*.ene"))
    if AF3_RESULTS_DIR.is_dir()
    else []
)
AF2_LOG_FILES = sorted(AF2_DIR.glob("*/log.txt")) if AF2_DIR.is_dir() else []
AF3_CONFIDENCE_FILES = (
    sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*_summary_confidences.json"))
    if AF3_RESULTS_DIR.is_dir()
    else []
)

raw_inventory = pd.DataFrame(
    [
        ("AF2 energy files found", len(AF2_ENERGY_FILES)),
        ("AF3 energy files found", len(AF3_ENERGY_FILES)),
        ("AF2 logs found", len(AF2_LOG_FILES)),
        ("AF3 confidence JSON found", len(AF3_CONFIDENCE_FILES)),
    ],
    columns=["item", "count"],
)
print(raw_inventory.to_string(index=False))


## 3. Helper functions and identifiers

The stable identifier is modelo_id, derived from the verified primary PDB name. AF2 and AF3 filenames are parsed separately because they encode different metadata.


In [ ]:
AF2_MODEL_RE = re.compile(
    rf"^(?P<case>{re.escape(CASE_ID)})_"
    r"(?P<state>unrelaxed|relaxed)_rank_(?P<rank>\d{3})_"
    r"(?P<af_version>alphafold2_multimer_v\d+)_"
    r"model_(?P<model_number>\d+)_seed_(?P<seed>\d+)"
    r"(?:\.r(?P<recycle>\d+))?$",
    re.IGNORECASE,
)
AF3_MODEL_RE = re.compile(
    rf"^(?P<case>{re.escape(CASE_ID)})_seed-(?P<seed>\d+)_"
    r"sample-(?P<sample>\d+)_model$",
    re.IGNORECASE,
)

def path_relative_to_repo(path):
    path = Path(path).resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()

def file_digest(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def files_are_identical(first, second):
    first, second = Path(first), Path(second)
    return (
        first.is_file()
        and second.is_file()
        and first.stat().st_size == second.stat().st_size
        and file_digest(first) == file_digest(second)
    )

def is_auxiliary_pdb(path):
    stem = Path(path).stem.lower()
    return stem.endswith("_rec") or stem.endswith("_lig")

def parse_model_metadata(model_path):
    model_path = Path(model_path)
    af2_match = AF2_MODEL_RE.fullmatch(model_path.stem)
    if af2_match:
        groups = af2_match.groupdict()
        return {
            "case_id": groups["case"].upper(),
            "modelo_id": model_path.stem,
            "prediction_method": "AF2",
            "af_version": groups["af_version"].lower(),
            "relaxation_status": groups["state"].lower(),
            "rank": int(groups["rank"]),
            "model_number": int(groups["model_number"]),
            "seed": int(groups["seed"]),
            "recycle": (
                int(groups["recycle"])
                if groups["recycle"] is not None
                else None
            ),
            "sample": None,
        }

    af3_match = AF3_MODEL_RE.fullmatch(model_path.stem)
    if af3_match:
        groups = af3_match.groupdict()
        return {
            "case_id": groups["case"].upper(),
            "modelo_id": model_path.stem,
            "prediction_method": "AF3",
            "af_version": "alphafold3",
            "relaxation_status": "as_predicted",
            "rank": None,
            "model_number": None,
            "seed": int(groups["seed"]),
            "recycle": None,
            "sample": int(groups["sample"]),
        }

    return None

def cohort_is_enabled(metadata):
    if metadata["prediction_method"] == "AF2":
        if metadata["relaxation_status"] == "unrelaxed":
            return INCLUDE_AF2_UNRELAXED
        if metadata["relaxation_status"] == "relaxed":
            return INCLUDE_AF2_RELAXED
        return False
    if metadata["prediction_method"] == "AF3":
        return INCLUDE_AF3_AS_PREDICTED
    return False


In [ ]:
def model_path_from_ini(energy_path):
    '''Resolve the original PDB from the pyDock ini file when possible.'''
    energy_path = Path(energy_path)
    ini_path = energy_path.with_suffix(".ini")
    if not ini_path.is_file():
        return None

    parser = configparser.ConfigParser(interpolation=None)
    parser.read(ini_path)
    for section in ("receptor", "ligand"):
        if not parser.has_option(section, "pdb"):
            continue
        value = parser.get(section, "pdb").strip().strip("\"'")
        candidate = Path(value).expanduser()
        if not candidate.is_absolute():
            candidate = ini_path.parent / candidate
        candidate = candidate.resolve()
        if candidate.is_file() and not is_auxiliary_pdb(candidate):
            return candidate
    return None

def resolve_model_path(energy_path):
    '''Use the companion ini first, then remove an explicit ligand suffix.'''
    energy_path = Path(energy_path)
    from_ini = model_path_from_ini(energy_path)
    if from_ini is not None:
        return from_ini, "ini"

    model_stem = re.sub(
        r"_LIG_[^.]+$",
        "",
        energy_path.stem,
        flags=re.IGNORECASE,
    )
    fallback = energy_path.with_name(f"{model_stem}.pdb").resolve()
    if fallback.is_file() and not is_auxiliary_pdb(fallback):
        return fallback, "validated_suffix"
    return None, None

def recycle20_energy_path(final_energy_path):
    final_energy_path = Path(final_energy_path)
    match = re.fullmatch(
        r"(?P<model>.+?)(?P<ligand>_LIG_[^.]+)",
        final_energy_path.stem,
        flags=re.IGNORECASE,
    )
    if not match:
        return None
    filename = (
        f"{match.group('model')}.r20{match.group('ligand')}"
        f"{final_energy_path.suffix}"
    )
    return final_energy_path.with_name(filename)

def duplicate_af2_final(model_path, energy_path=None):
    '''Return true only when an unsuffixed AF2 final duplicates recycle 20.'''
    model_path = Path(model_path)
    metadata = parse_model_metadata(model_path)
    if (
        metadata is None
        or metadata["prediction_method"] != "AF2"
        or metadata["recycle"] is not None
    ):
        return False, None

    recycle_model = model_path.with_name(f"{model_path.stem}.r20.pdb")
    if not files_are_identical(model_path, recycle_model):
        return False, None

    if energy_path is not None:
        recycle_energy = recycle20_energy_path(energy_path)
        if recycle_energy is None or not files_are_identical(
            energy_path, recycle_energy
        ):
            return False, None

    return True, recycle_model


In [ ]:
ENE_ENERGY_COLUMNS = ["Ele", "Desolv", "VDW", "Total"]

def read_energy_file(energy_path):
    table = pd.read_csv(energy_path, sep=r"\s+", skiprows=[1])
    missing = [
        column for column in ENE_ENERGY_COLUMNS if column not in table
    ]
    if missing:
        raise ValueError(f"Missing pyDock columns {missing}")
    if table.empty:
        raise ValueError("Empty pyDock energy table")

    if len(table) > 1 and "RANK" in table:
        rank_values = pd.to_numeric(table["RANK"], errors="coerce")
        row = table.loc[rank_values.sort_values().index[0]]
    else:
        row = table.iloc[0]

    values = {
        column: pd.to_numeric(row[column], errors="coerce")
        for column in ("Ele", "Desolv", "VDW")
    }
    # Normalize the native Total field without modifying the source .ene.
    values["pydock"] = pd.to_numeric(row["Total"], errors="coerce")
    return values

def load_pydock_energies(energy_paths):
    diagnostics = {
        "associated_pdb": [],
        "missing_pdb": [],
        "unrecognized_model": [],
        "excluded_duplicate": [],
        "excluded_by_flags": [],
        "read_errors": [],
    }
    records = []

    for energy_path in sorted(map(Path, energy_paths)):
        model_path, resolution = resolve_model_path(energy_path)
        if model_path is None:
            diagnostics["missing_pdb"].append(
                {"path": path_relative_to_repo(energy_path), "detail": ""}
            )
            continue

        diagnostics["associated_pdb"].append(
            {"path": path_relative_to_repo(energy_path), "detail": resolution}
        )
        metadata = parse_model_metadata(model_path)
        if metadata is None:
            diagnostics["unrecognized_model"].append(
                {
                    "path": path_relative_to_repo(energy_path),
                    "detail": path_relative_to_repo(model_path),
                }
            )
            continue

        if not cohort_is_enabled(metadata):
            diagnostics["excluded_by_flags"].append(
                {
                    "path": path_relative_to_repo(energy_path),
                    "detail": metadata["relaxation_status"],
                }
            )
            continue

        if (
            DROP_DUPLICATE_AF2_FINALS
            and metadata["prediction_method"] == "AF2"
            and metadata["recycle"] is None
        ):
            is_duplicate, recycle_model = duplicate_af2_final(
                model_path, energy_path
            )
            if is_duplicate:
                diagnostics["excluded_duplicate"].append(
                    {
                        "path": path_relative_to_repo(energy_path),
                        "detail": path_relative_to_repo(recycle_model),
                    }
                )
                continue

        try:
            energy_values = read_energy_file(energy_path)
        except Exception as exc:
            diagnostics["read_errors"].append(
                {
                    "path": path_relative_to_repo(energy_path),
                    "detail": str(exc),
                }
            )
            continue

        records.append(
            {
                **metadata,
                "model_path": path_relative_to_repo(model_path),
                "energy_path": path_relative_to_repo(energy_path),
                "pdb_resolution": resolution,
                **energy_values,
            }
        )

    frame = pd.DataFrame(records)
    if not frame.empty:
        key = ["case_id", "prediction_method", "modelo_id"]
        duplicated = frame.duplicated(key, keep=False)
        if duplicated.any():
            examples = frame.loc[duplicated, key + ["energy_path"]]
            raise ValueError(
                "More than one pyDock energy file maps to a model:\n"
                + examples.to_string(index=False)
            )
    return frame, diagnostics


In [ ]:
def inventory_primary_models():
    records = []
    diagnostics = {
        "excluded_duplicate": [],
        "excluded_by_flags": [],
        "ignored_auxiliary": [],
    }

    af2_pdbs = sorted(AF2_DIR.rglob("*.pdb")) if AF2_DIR.is_dir() else []
    for model_path in af2_pdbs:
        if is_auxiliary_pdb(model_path):
            diagnostics["ignored_auxiliary"].append(
                {"path": path_relative_to_repo(model_path), "detail": ""}
            )
            continue
        metadata = parse_model_metadata(model_path)
        if metadata is None:
            continue
        if not cohort_is_enabled(metadata):
            diagnostics["excluded_by_flags"].append(
                {
                    "path": path_relative_to_repo(model_path),
                    "detail": metadata["relaxation_status"],
                }
            )
            continue

        if DROP_DUPLICATE_AF2_FINALS and metadata["recycle"] is None:
            energy_candidates = sorted(
                model_path.parent.glob(f"{model_path.stem}_LIG_*.ene")
            )
            if energy_candidates:
                duplicate, recycle_model = duplicate_af2_final(
                    model_path, energy_candidates[0]
                )
            else:
                duplicate, recycle_model = duplicate_af2_final(model_path)
            if duplicate:
                diagnostics["excluded_duplicate"].append(
                    {
                        "path": path_relative_to_repo(model_path),
                        "detail": path_relative_to_repo(recycle_model),
                    }
                )
                continue

        records.append(
            {**metadata, "model_path": path_relative_to_repo(model_path)}
        )

    af3_pdbs = (
        sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*_model.pdb"))
        if AF3_RESULTS_DIR.is_dir()
        else []
    )
    for model_path in af3_pdbs:
        metadata = parse_model_metadata(model_path)
        if metadata is None:
            continue
        if not cohort_is_enabled(metadata):
            diagnostics["excluded_by_flags"].append(
                {
                    "path": path_relative_to_repo(model_path),
                    "detail": metadata["relaxation_status"],
                }
            )
            continue
        records.append(
            {**metadata, "model_path": path_relative_to_repo(model_path)}
        )

    frame = pd.DataFrame(records)
    if not frame.empty:
        key = ["case_id", "prediction_method", "modelo_id"]
        if frame.duplicated(key).any():
            raise ValueError("Duplicate primary model identifiers were found")
    return frame, diagnostics


## 4. Confidence readers

AF2 confidence is read from recycle entries in each ColabFold log. The notebook never overrides the structural state: current 4POU AF2 models remain labelled unrelaxed.


In [ ]:
FLOAT_PATTERN = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"
AF2_LOG_RE = re.compile(
    rf"(?P<af_version>alphafold2_multimer_v\d+)_"
    rf"model_(?P<model_number>\d+)_seed_(?P<seed>\d+)\s+"
    rf"recycle=(?P<recycle>\d+)\s+"
    rf"pLDDT=(?P<plddt>{FLOAT_PATTERN})\s+"
    rf"pTM=(?P<ptm>{FLOAT_PATTERN})\s+"
    rf"ipTM=(?P<iptm>{FLOAT_PATTERN})"
    rf"(?:\s+tol=(?P<tol>{FLOAT_PATTERN}))?"
)

def load_af2_confidence(log_paths):
    records = []
    for log_path in sorted(map(Path, log_paths)):
        with log_path.open(encoding="utf-8", errors="replace") as handle:
            for line in handle:
                match = AF2_LOG_RE.search(line)
                if not match:
                    continue
                groups = match.groupdict()
                records.append(
                    {
                        "case_id": CASE_ID,
                        "prediction_method": "AF2",
                        "af_version": groups["af_version"].lower(),
                        "model_number": int(groups["model_number"]),
                        "seed": int(groups["seed"]),
                        "recycle": int(groups["recycle"]),
                        "plddt": float(groups["plddt"]),
                        "ptm": float(groups["ptm"]),
                        "iptm": float(groups["iptm"]),
                        "ranking_score": np.nan,
                        "tol": (
                            float(groups["tol"])
                            if groups["tol"] is not None
                            else np.nan
                        ),
                        "confidence_path": path_relative_to_repo(log_path),
                    }
                )

    frame = pd.DataFrame(records)
    if not frame.empty:
        keys = [
            "case_id",
            "prediction_method",
            "af_version",
            "model_number",
            "seed",
            "recycle",
        ]
        duplicated = frame.duplicated(keys, keep=False)
        if duplicated.any():
            raise ValueError(
                "Duplicate AF2 confidence records:\n"
                + frame.loc[duplicated, keys].to_string(index=False)
            )
    return frame


Local AF3 confidence is read only from summary JSON files inside seed/sample directories. The root aggregate summary is deliberately ignored. pLDDT is left unavailable because it is not present in these summary files.


In [ ]:
AF3_SUMMARY_RE = re.compile(
    rf"^(?P<case>{re.escape(CASE_ID)})_seed-(?P<seed>\d+)_"
    r"sample-(?P<sample>\d+)_summary_confidences$",
    re.IGNORECASE,
)

def json_for_csv(value):
    if value is None:
        return None
    return json.dumps(value, separators=(",", ":"))

def load_af3_confidence(summary_paths):
    records = []
    issues = []

    for summary_path in sorted(map(Path, summary_paths)):
        match = AF3_SUMMARY_RE.fullmatch(summary_path.stem)
        if not match:
            issues.append(
                {
                    "path": path_relative_to_repo(summary_path),
                    "detail": "unrecognized summary filename",
                }
            )
            continue

        try:
            payload = json.loads(summary_path.read_text(encoding="utf-8"))
        except Exception as exc:
            issues.append(
                {
                    "path": path_relative_to_repo(summary_path),
                    "detail": str(exc),
                }
            )
            continue

        groups = match.groupdict()
        seed = int(groups["seed"])
        sample = int(groups["sample"])
        modelo_id = f"{groups['case']}_seed-{seed}_sample-{sample}_model"

        chain_ids = list(dict.fromkeys(payload.get("chain_ids", [])))
        chain_ptm = payload.get("chain_ptm")
        chain_pair_iptm = payload.get("chain_pair_iptm")

        chain_a_ptm = np.nan
        chain_b_ptm = np.nan
        iptm_a_vs_b = np.nan
        if isinstance(chain_ptm, list):
            if len(chain_ptm) > 0:
                chain_a_ptm = chain_ptm[0]
            if len(chain_ptm) > 1:
                chain_b_ptm = chain_ptm[1]
        if (
            isinstance(chain_pair_iptm, list)
            and len(chain_pair_iptm) > 0
            and isinstance(chain_pair_iptm[0], list)
            and len(chain_pair_iptm[0]) > 1
        ):
            iptm_a_vs_b = chain_pair_iptm[0][1]

        records.append(
            {
                "case_id": groups["case"].upper(),
                "modelo_id": modelo_id,
                "prediction_method": "AF3",
                "af_version": "alphafold3",
                "seed": seed,
                "sample": sample,
                "plddt": np.nan,
                "ptm": pd.to_numeric(payload.get("ptm"), errors="coerce"),
                "iptm": pd.to_numeric(payload.get("iptm"), errors="coerce"),
                "ranking_score": pd.to_numeric(
                    payload.get("ranking_score"), errors="coerce"
                ),
                "fraction_disordered": pd.to_numeric(
                    payload.get("fraction_disordered"), errors="coerce"
                ),
                "has_clash": payload.get("has_clash"),
                "num_recycles": payload.get("num_recycles"),
                "chain_ids": json_for_csv(chain_ids),
                "chain_ptm": json_for_csv(chain_ptm),
                "chain_pair_iptm": json_for_csv(chain_pair_iptm),
                "chain_A_ptm": chain_a_ptm,
                "chain_B_ptm": chain_b_ptm,
                "iptm_A_vs_B": iptm_a_vs_b,
                "confidence_path": path_relative_to_repo(summary_path),
            }
        )

    frame = pd.DataFrame(records)
    if not frame.empty:
        keys = ["case_id", "prediction_method", "modelo_id", "seed", "sample"]
        duplicated = frame.duplicated(keys, keep=False)
        if duplicated.any():
            raise ValueError(
                "Duplicate AF3 confidence records:\n"
                + frame.loc[duplicated, keys].to_string(index=False)
            )
    return frame, issues


In [ ]:
def safe_zscore(series):
    '''Return population Z-scores; a constant valid cohort receives zero.'''
    numeric = pd.to_numeric(series, errors="coerce")
    result = pd.Series(np.nan, index=series.index, dtype=float)
    valid = numeric.dropna()
    if valid.empty:
        return result

    standard_deviation = valid.std(ddof=0)
    if not np.isfinite(standard_deviation) or np.isclose(
        standard_deviation, 0.0
    ):
        result.loc[valid.index] = 0.0
    else:
        result.loc[valid.index] = (
            valid - valid.mean()
        ) / standard_deviation
    return result


## 5. Load pyDock energies and the selected model cohort

The native Total field is read from each pyDock energy file and normalized to the table column pydock; the source .ene remains unchanged. This score corresponds to Ele + Desolv + 0.1 × VDW. The full van der Waals score, pydock_1vdw, is calculated explicitly as Ele + Desolv + VDW. Both metrics are retained and ranked across all selected models in each case.


In [ ]:
model_inventory, inventory_diagnostics = inventory_primary_models()
energy_df, energy_diagnostics = load_pydock_energies(
    [*AF2_ENERGY_FILES, *AF3_ENERGY_FILES]
)

print(f"Primary models selected : {len(model_inventory)}")
print(f"Models with pyDock data : {len(energy_df)}")
if not energy_df.empty:
    print(
        energy_df.groupby(
            ["prediction_method", "relaxation_status"]
        ).size().to_string()
    )


## 6. Load AF2 confidence


In [ ]:
af2_confidence = load_af2_confidence(AF2_LOG_FILES)
print(f"AF2 confidence records: {len(af2_confidence)}")
if not af2_confidence.empty:
    print(
        af2_confidence.groupby("af_version").size().to_string()
    )


In [ ]:
af2_confidence

## 7. Load local AF3 confidence


In [ ]:
af3_confidence, af3_confidence_issues = load_af3_confidence(
    AF3_CONFIDENCE_FILES
)
print(f"AF3 confidence records: {len(af3_confidence)}")
if af3_confidence_issues:
    print(pd.DataFrame(af3_confidence_issues).to_string(index=False))


## 8. Explicit, validated merges

AF2 uses version, model number, seed and recycle as scientific identifiers. AF3 uses modelo_id, seed and sample. Only records with a verified PDB, pyDock energy and matching confidence record continue to the consolidated table.


In [ ]:
af2_energy = energy_df.loc[
    energy_df["prediction_method"].eq("AF2")
].copy()
af3_energy = energy_df.loc[
    energy_df["prediction_method"].eq("AF3")
].copy()

AF2_MERGE_KEYS = [
    "case_id",
    "prediction_method",
    "af_version",
    "model_number",
    "seed",
    "recycle",
]
AF3_MERGE_KEYS = [
    "case_id",
    "prediction_method",
    "af_version",
    "modelo_id",
    "seed",
    "sample",
]

if af2_energy.empty and af2_confidence.empty:
    af2_merged = pd.DataFrame()
    af2_merge_counts = {}
else:
    af2_merged = af2_energy.merge(
        af2_confidence,
        how="outer",
        on=AF2_MERGE_KEYS,
        validate="many_to_one",
        indicator="merge_status",
    )
    af2_merge_counts = af2_merged["merge_status"].value_counts().to_dict()

if af3_energy.empty and af3_confidence.empty:
    af3_merged = pd.DataFrame()
    af3_merge_counts = {}
else:
    af3_merged = af3_energy.merge(
        af3_confidence,
        how="outer",
        on=AF3_MERGE_KEYS,
        validate="one_to_one",
        indicator="merge_status",
    )
    af3_merge_counts = af3_merged["merge_status"].value_counts().to_dict()

matched_frames = []
if not af2_merged.empty:
    matched_frames.append(
        af2_merged.loc[af2_merged["merge_status"].eq("both")]
        .drop(columns="merge_status")
        .copy()
    )
if not af3_merged.empty:
    matched_frames.append(
        af3_merged.loc[af3_merged["merge_status"].eq("both")]
        .drop(columns="merge_status")
        .copy()
    )

if not matched_frames:
    raise ValueError("No model has both pyDock energy and confidence data")

consolidated = pd.concat(matched_frames, ignore_index=True, sort=False)
model_key = ["case_id", "prediction_method", "modelo_id"]
if consolidated.duplicated(model_key).any():
    raise ValueError("The consolidated model identifier is not unique")

print(f"AF2 merge status: {af2_merge_counts}")
print(f"AF3 merge status: {af3_merge_counts}")
print(f"Models successfully merged: {len(consolidated)}")


## 9. Confidence, normalization and combined ranking

Model confidence is 0.8 × ipTM + 0.2 × pTM. The notebook calculates independent global rankings and Z-scores for pydock and pydock_1vdw. The main combined score uses pyDock-1VDW:

**combined_score = model_confidence_z - pydock_1vdw_z**

All normalizations and energy rankings are calculated across the selected cohort for each case. Missing metrics remain missing; a zero-variance metric receives a Z-score of zero rather than causing an error.


In [ ]:
consolidated["model_confidence"] = (
    0.8 * pd.to_numeric(consolidated["iptm"], errors="coerce")
    + 0.2 * pd.to_numeric(consolidated["ptm"], errors="coerce")
)
consolidated["pydock_1vdw"] = (
    pd.to_numeric(consolidated["Ele"], errors="coerce")
    + pd.to_numeric(consolidated["Desolv"], errors="coerce")
    + pd.to_numeric(consolidated["VDW"], errors="coerce")
)

consolidated["pydock_rank"] = (
    consolidated.groupby("case_id")["pydock"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)
consolidated["pydock_1vdw_rank"] = (
    consolidated.groupby("case_id")["pydock_1vdw"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)

consolidated["model_confidence_z"] = consolidated.groupby(
    "case_id", group_keys=False
)["model_confidence"].transform(safe_zscore)
consolidated["pydock_z"] = consolidated.groupby(
    "case_id", group_keys=False
)["pydock"].transform(safe_zscore)
consolidated["pydock_1vdw_z"] = consolidated.groupby(
    "case_id", group_keys=False
)["pydock_1vdw"].transform(safe_zscore)
consolidated["combined_score"] = (
    consolidated["model_confidence_z"]
    - consolidated["pydock_1vdw_z"]
)

required_for_ranking = [
    "model_path",
    "energy_path",
    "confidence_path",
    "model_confidence",
    "pydock",
    "pydock_1vdw",
    "pydock_rank",
    "pydock_1vdw_rank",
    "model_confidence_z",
    "pydock_z",
    "pydock_1vdw_z",
    "combined_score",
]
rankable = consolidated[required_for_ranking].notna().all(axis=1)

ranking = consolidated.loc[rankable].copy()
ranking = ranking.sort_values(
    [
        "combined_score",
        "model_confidence",
        "pydock_1vdw",
        "pydock",
        "modelo_id",
    ],
    ascending=[False, False, True, True, True],
    kind="mergesort",
).reset_index(drop=True)
ranking["combined_rank"] = pd.Series(
    np.arange(1, len(ranking) + 1), dtype="Int64"
)

rank_lookup = ranking[
    model_key + ["combined_rank"]
]
consolidated = consolidated.merge(
    rank_lookup,
    how="left",
    on=model_key,
    validate="one_to_one",
)
consolidated["combined_rank"] = consolidated["combined_rank"].astype("Int64")

COLUMN_ORDER = [
    "case_id",
    "modelo_id",
    "prediction_method",
    "af_version",
    "relaxation_status",
    "model_path",
    "energy_path",
    "confidence_path",
    "Ele",
    "Desolv",
    "VDW",
    "pydock",
    "pydock_1vdw",
    "pydock_rank",
    "pydock_1vdw_rank",
    "plddt",
    "ptm",
    "iptm",
    "ranking_score",
    "model_confidence",
    "model_confidence_z",
    "pydock_z",
    "pydock_1vdw_z",
    "combined_score",
    "combined_rank",
]
EXTRA_COLUMNS = [
    "rank",
    "model_number",
    "seed",
    "recycle",
    "sample",
    "tol",
    "fraction_disordered",
    "has_clash",
    "num_recycles",
    "chain_ids",
    "chain_ptm",
    "chain_pair_iptm",
    "chain_A_ptm",
    "chain_B_ptm",
    "iptm_A_vs_B",
    "pdb_resolution",
]
export_columns = [
    column
    for column in [*COLUMN_ORDER, *EXTRA_COLUMNS]
    if column in consolidated.columns
]

consolidated_export = consolidated[export_columns].sort_values(
    ["prediction_method", "af_version", "modelo_id"],
    kind="mergesort",
)
ranking_export = ranking[
    [column for column in export_columns if column in ranking.columns]
].sort_values("combined_rank", kind="mergesort")

INTEGER_EXPORT_COLUMNS = [
    "pydock_rank",
    "pydock_1vdw_rank",
    "combined_rank",
    "rank",
    "model_number",
    "seed",
    "recycle",
    "sample",
    "num_recycles",
]
for export_frame in (consolidated_export, ranking_export):
    for column in INTEGER_EXPORT_COLUMNS:
        if column in export_frame.columns:
            export_frame[column] = pd.to_numeric(
                export_frame[column], errors="coerce"
            ).astype("Int64")

consolidated_path = ANALYSIS_DIR / "consolidated_results.csv"
ranking_path = ANALYSIS_DIR / "combined_ranking.csv"
consolidated_export.to_csv(consolidated_path, index=False)
ranking_export.to_csv(ranking_path, index=False)

print(f"Consolidated table: {consolidated_path}")
print(f"Combined ranking  : {ranking_path}")
print(f"Ranked models     : {len(ranking_export)}")
print(ranking_export[COLUMN_ORDER].head(10).to_string(index=False))


## 10. Lightweight plots

The plots compare the selected cohort only. They are written beside the CSV files and do not trigger AlphaFold, relaxation or pyDock calculations.


In [ ]:
plot_paths = []

if GENERATE_PLOTS and not ranking_export.empty:
    methods = list(ranking_export["prediction_method"].dropna().unique())
    colours = {"AF2": "#377eb8", "AF3": "#e41a1c"}
    metric_labels = {
        "model_confidence": "Model confidence",
        "pydock": "pyDock",
        "pydock_1vdw": "pyDock-1VDW",
        "combined_score": "Combined score",
    }

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for axis, column in zip(axes.flatten(), metric_labels):
        for method in methods:
            values = ranking_export.loc[
                ranking_export["prediction_method"].eq(method), column
            ].dropna()
            if not values.empty:
                axis.hist(
                    values,
                    bins=min(25, max(5, int(np.sqrt(len(values))))),
                    alpha=0.55,
                    label=method,
                    color=colours.get(method),
                )
        axis.set_title(metric_labels[column])
        axis.set_ylabel("Models")
        axis.legend()
    fig.tight_layout()
    distribution_path = ANALYSIS_DIR / "metric_distributions.png"
    fig.savefig(distribution_path, dpi=160)
    plt.close(fig)
    plot_paths.append(distribution_path)

    fig, axis = plt.subplots(figsize=(7, 4.5))
    zscore_plot_data = [
        ("model_confidence_z", "Model confidence Z-score", "#4daf4a"),
        ("pydock_z", "pyDock Z-score", "#984ea3"),
        ("pydock_1vdw_z", "pyDock-1VDW Z-score", "#377eb8"),
        ("combined_score", "Combined score", "#ff7f00"),
    ]
    for column, label, colour in zscore_plot_data:
        values = ranking_export[column].dropna()
        if not values.empty:
            axis.hist(
                values,
                bins=min(30, max(6, int(np.sqrt(len(values))))),
                density=True,
                histtype="step",
                linewidth=2,
                label=label,
                color=colour,
            )
    axis.set_xlabel("Metric value")
    axis.set_ylabel("Density")
    axis.legend()
    fig.tight_layout()
    density_path = ANALYSIS_DIR / "metric_density.png"
    fig.savefig(density_path, dpi=160)
    plt.close(fig)
    plot_paths.append(density_path)

    fig, axis = plt.subplots(figsize=(7, 5))
    for method in methods:
        subset = ranking_export.loc[
            ranking_export["prediction_method"].eq(method)
        ]
        axis.scatter(
            subset["pydock_1vdw"],
            subset["model_confidence"],
            s=24,
            alpha=0.75,
            label=method,
            color=colours.get(method),
        )
    axis.set_xlabel("pyDock-1VDW (lower is better)")
    axis.set_ylabel("Model confidence (higher is better)")
    axis.legend()
    fig.tight_layout()
    scatter_path = ANALYSIS_DIR / "confidence_vs_pydock.png"
    fig.savefig(scatter_path, dpi=160)
    plt.close(fig)
    plot_paths.append(scatter_path)

for path in plot_paths:
    print(path)


## 11. Diagnostics

These counts expose exclusions instead of silently changing the cohort. A count different from the expected 315 unique AF2 plus 5 local AF3 models is reported but does not by itself stop the notebook.


In [ ]:
inventory_ids = set(
    zip(
        model_inventory.get("case_id", pd.Series(dtype=str)),
        model_inventory.get("prediction_method", pd.Series(dtype=str)),
        model_inventory.get("modelo_id", pd.Series(dtype=str)),
    )
)
energy_ids = set(
    zip(
        energy_df.get("case_id", pd.Series(dtype=str)),
        energy_df.get("prediction_method", pd.Series(dtype=str)),
        energy_df.get("modelo_id", pd.Series(dtype=str)),
    )
)
merged_ids = set(
    zip(
        consolidated.get("case_id", pd.Series(dtype=str)),
        consolidated.get("prediction_method", pd.Series(dtype=str)),
        consolidated.get("modelo_id", pd.Series(dtype=str)),
    )
)

models_missing_energy = inventory_ids - energy_ids
models_missing_confidence = energy_ids - merged_ids
confidence_without_energy = (
    int(af2_merge_counts.get("right_only", 0))
    + int(af3_merge_counts.get("right_only", 0))
)

diagnostic_table = pd.DataFrame(
    [
        ("AF2 energy files found", len(AF2_ENERGY_FILES)),
        ("AF3 energy files found", len(AF3_ENERGY_FILES)),
        ("AF2 logs found", len(AF2_LOG_FILES)),
        ("AF3 confidence JSON found", len(AF3_CONFIDENCE_FILES)),
        (
            "Energy files with associated PDB",
            len(energy_diagnostics["associated_pdb"]),
        ),
        ("Primary models selected", len(model_inventory)),
        ("Models with energy", len(energy_df)),
        (
            "Models with confidence",
            len(af2_confidence) + len(af3_confidence),
        ),
        ("Models successfully merged", len(consolidated)),
        (
            "Models excluded as duplicate AF2 finals",
            len(energy_diagnostics["excluded_duplicate"]),
        ),
        (
            "Models excluded by cohort flags",
            len(energy_diagnostics["excluded_by_flags"]),
        ),
        ("Energy files missing PDB", len(energy_diagnostics["missing_pdb"])),
        ("Primary models missing energy", len(models_missing_energy)),
        ("Energy models missing confidence", len(models_missing_confidence)),
        ("Confidence records missing energy", confidence_without_energy),
        ("Models excluded for incomplete ranking metrics", int((~rankable).sum())),
        ("Final ranked models", len(ranking_export)),
    ],
    columns=["diagnostic", "count"],
)
print(diagnostic_table.to_string(index=False))

issue_rows = []
for category in (
    "missing_pdb",
    "unrecognized_model",
    "excluded_duplicate",
    "excluded_by_flags",
    "read_errors",
):
    for issue in energy_diagnostics[category]:
        issue_rows.append({"category": category, **issue})
for issue in af3_confidence_issues:
    issue_rows.append({"category": "af3_confidence", **issue})

if issue_rows:
    print("\nFile-level diagnostics (first 20):")
    print(pd.DataFrame(issue_rows).head(20).to_string(index=False))
else:
    print("\nNo file-level input issues were detected.")

EXPECTED_DEFAULT_RANKED_MODELS = 320
if len(ranking_export) != EXPECTED_DEFAULT_RANKED_MODELS:
    print(
        "\nNote: the final count differs from the current 4POU expectation "
        f"({EXPECTED_DEFAULT_RANKED_MODELS}). Review the diagnostics above."
    )


## 12. Scope and future extension

This workflow deliberately has no dependency on an experimental structure. RMSD against a reference is postponed as an optional, separate table that could later be joined through case_id, prediction_method and modelo_id without changing the primary confidence–energy ranking.
